# Customer Risk Profiling and Anomaly Detection System
## Banking Domain | Unsupervised ML

**Objective:**
Banks manage thousands of credit card customers with vastly different behavioural profiles. A one-size-fits-all risk or marketing strategy is inefficient. This project builds a data-driven system that:
1. Segments customers into behavioural archetypes using unsupervised clustering
2. Detects anomalous spend behaviour *within each segment* — flagging customers who deviate from their own peer group rather than the global population

The per-segment anomaly approach is the core design decision here. A high-balance customer is normal in the Revolver segment but anomalous in the Inactive segment — global anomaly detection misses this distinction entirely.

**Dataset:** CC_GENERAL.csv — 8,950 active credit card holders, 18 behavioural variables, 6-month observation window.

**Stack:** Python · scikit-learn · pandas · Plotly · Seaborn

---
## 1. Environment Setup

In [ ]:
!pip install -q plotly

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score, silhouette_samples

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
pd.set_option('display.float_format', '{:.4f}'.format)

print('Environment ready.')

---
## 2. Data Loading and Initial Assessment

First pass: understand what we're working with before touching a single value.

In [ ]:
# Upload CC_GENERAL.csv via the Files panel before running this cell
df = pd.read_csv('CC_GENERAL.csv')

print(f'Dataset: {df.shape[0]} customers × {df.shape[1]} features')
df.head()

In [ ]:
# Schema and missing value audit
print('=== SCHEMA ===')
df.info()
print()

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'Count': missing, 'Pct': missing_pct})
print('=== MISSING VALUES ===')
print(missing_report[missing_report['Count'] > 0])

---
## 3. Exploratory Data Analysis

Two things I want to establish here: the shape of each feature's distribution, and which features carry redundant information. Both inform preprocessing and model design decisions downstream.

In [ ]:
# Feature distributions
# Most financial features here are heavily right-skewed —
# a small number of high-value customers dominate the upper tail.
# This rules out treating raw values as cluster inputs directly.

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

fig, axes = plt.subplots(nrows=4, ncols=5, figsize=(20, 14))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col].dropna(), bins=50, color='steelblue', alpha=0.7, edgecolor='white')
    axes[i].set_title(col, fontsize=9, fontweight='bold')
    axes[i].tick_params(labelsize=7)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions — All Variables', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation structure
# High correlation between purchase-related variables confirms
# that PCA will be effective at compressing this feature space
# without significant information loss.

plt.figure(figsize=(16, 12))
corr_matrix = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, annot_kws={'size': 7}
)
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

# Top correlated pairs
corr_pairs = corr_matrix.unstack().sort_values(ascending=False)
corr_pairs = corr_pairs[corr_pairs < 1.0]
print('Top correlated feature pairs:')
print(corr_pairs.head(10)[::2].to_string())

---
## 4. Preprocessing Pipeline

Three decisions here, each made deliberately:

**Missing values → median imputation.** MINIMUM_PAYMENTS has ~313 missing entries. Given the right skew of the distribution, median is a more stable central estimate than mean.

**Skewed features → log(x+1) transform.** Brings the heavy right tails into a range where Euclidean distance (which K-Means relies on) is meaningful. Without this, a single extreme spender distorts the entire cluster geometry.

**All features → StandardScaler.** K-Means is distance-based. An unscaled MINIMUM_PAYMENTS (range: 0–76k) would dominate an unscaled TENURE (range: 6–12) purely by magnitude. Scaling puts every feature on the same footing.

In [ ]:
# Drop customer ID — identifier, not a feature
if 'CUST_ID' in df.columns:
    customer_ids = df['CUST_ID'].copy()
    df_clean = df.drop(columns=['CUST_ID'])
else:
    customer_ids = df.index
    df_clean = df.copy()

# Median imputation
df_clean = df_clean.fillna(df_clean.median(numeric_only=True))
print(f'Missing values after imputation: {df_clean.isnull().sum().sum()}')

In [ ]:
# Log transform — applied to monetary/count features only
# Frequency and ratio features (already bounded 0-1) are excluded
skip_log = [
    'BALANCE_FREQUENCY', 'PURCHASES_FREQUENCY', 'ONEOFF_PURCHASES_FREQUENCY',
    'PURCHASES_INSTALLMENTS_FREQUENCY', 'CASH_ADVANCE_FREQUENCY',
    'PRC_FULL_PAYMENT', 'TENURE'
]

df_log = df_clean.copy()
log_cols = [c for c in df_clean.columns if c not in skip_log]
for col in log_cols:
    df_log[col] = np.log1p(df_log[col])

# StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_log)
X_scaled_df = pd.DataFrame(X_scaled, columns=df_clean.columns)

print(f'Preprocessed feature matrix: {X_scaled_df.shape}')

---
## 5. KPI Engineering

Raw features capture what happened. Derived KPIs capture *how* a customer behaves. The distinction matters for segmentation — two customers with identical purchase amounts but different credit limits represent different risk profiles. I engineered six ratio-based KPIs to capture these behavioural dimensions.

In [ ]:
df_kpi = df_clean.copy()

# Credit utilisation intensity — how aggressively the customer uses available credit
df_kpi['PURCHASE_TO_LIMIT'] = df_kpi['PURCHASES'] / df_kpi['CREDIT_LIMIT'].replace(0, 1)

# Cash dependency ratio — fraction of total outflows taken as cash advance
# High values indicate short-term liquidity stress or high-risk behaviour
total_outflow = df_kpi['PURCHASES'] + df_kpi['CASH_ADVANCE']
df_kpi['CASH_ADVANCE_RATIO'] = df_kpi['CASH_ADVANCE'] / total_outflow.replace(0, 1)

# Payment discipline ratio — payments made vs minimum required
# Values > 1 indicate responsible payers; values < 1 flag missed minimums
df_kpi['PAYMENT_TO_MINPAYMENT'] = df_kpi['PAYMENTS'] / df_kpi['MINIMUM_PAYMENTS'].replace(0, 1)

# Instalment preference — structured vs impulse purchase behaviour
df_kpi['INSTALLMENT_RATIO'] = df_kpi['INSTALLMENTS_PURCHASES'] / df_kpi['PURCHASES'].replace(0, 1)

# Average transaction value — spending pattern (big-ticket vs micro-transactions)
df_kpi['SPEND_PER_TRANSACTION'] = df_kpi['PURCHASES'] / df_kpi['PURCHASES_TRX'].replace(0, 1)

# Monthly balance burden — average balance carried per active month
df_kpi['MONTHLY_AVG_BALANCE'] = df_kpi['BALANCE'] / df_kpi['TENURE'].replace(0, 1)

kpi_cols = ['PURCHASE_TO_LIMIT', 'CASH_ADVANCE_RATIO', 'PAYMENT_TO_MINPAYMENT',
            'INSTALLMENT_RATIO', 'SPEND_PER_TRANSACTION', 'MONTHLY_AVG_BALANCE']

print(f'KPIs engineered: {len(kpi_cols)}')
print(df_kpi[kpi_cols].describe())

In [ ]:
# Clean and scale the full enriched feature set
df_kpi.replace([np.inf, -np.inf], np.nan, inplace=True)
df_kpi.fillna(df_kpi.median(numeric_only=True), inplace=True)

skip_log_kpi = skip_log + ['PURCHASE_TO_LIMIT', 'CASH_ADVANCE_RATIO', 'INSTALLMENT_RATIO']
df_kpi_log = df_kpi.copy()
for col in df_kpi_log.columns:
    if col not in skip_log_kpi and col != 'CLUSTER':
        df_kpi_log[col] = np.log1p(df_kpi_log[col].clip(lower=0))

feature_cols = [c for c in df_kpi_log.columns if c != 'CLUSTER']
scaler_kpi = StandardScaler()
X_kpi = scaler_kpi.fit_transform(df_kpi_log[feature_cols])
X_kpi_df = pd.DataFrame(X_kpi, columns=feature_cols)

print(f'Final feature matrix: {X_kpi_df.shape} — {len(feature_cols)} features per customer')

---
## 6. Dimensionality Reduction via PCA

With 23 features, several of which are correlated, I use PCA to compress to a lower-dimensional representation before clustering. Two versions:
- **Clustering PCA:** retain enough components to explain 80% of variance — preserves most signal while reducing noise
- **Visualisation PCA:** 2 components only, purely for plotting the cluster structure

In [ ]:
# Determine component count via explained variance analysis
pca_full = PCA(random_state=42)
pca_full.fit(X_kpi)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(range(1, len(pca_full.explained_variance_ratio_)+1),
        pca_full.explained_variance_ratio_ * 100,
        color='steelblue', alpha=0.8, edgecolor='white')
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Explained Variance (%)')
ax1.set_title('Variance per Component')
ax1.set_xlim(0.5, 15)

ax2.plot(range(1, len(cumvar)+1), cumvar * 100, 'b-o', markersize=5)
ax2.axhline(y=80, color='red', linestyle='--', label='80% threshold')
ax2.axhline(y=90, color='orange', linestyle='--', label='90% threshold')
ax2.set_xlabel('Number of Components')
ax2.set_ylabel('Cumulative Variance (%)')
ax2.set_title('Cumulative Explained Variance')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('PCA Variance Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('pca_variance.png', dpi=120, bbox_inches='tight')
plt.show()

n_components = np.argmax(cumvar >= 0.80) + 1
print(f'Components for 80% variance: {n_components}')

In [ ]:
N_COMPONENTS = n_components

pca = PCA(n_components=N_COMPONENTS, random_state=42)
X_pca = pca.fit_transform(X_kpi)

pca_2d = PCA(n_components=2, random_state=42)
X_2d = pca_2d.fit_transform(X_kpi)

print(f'Clustering input: {X_pca.shape}  ({pca.explained_variance_ratio_.sum()*100:.1f}% variance retained)')
print(f'Visualisation input: {X_2d.shape}')

---
## 7. Clustering — Selecting Optimal K

I evaluate K from 2 to 10 using two complementary metrics:

**Inertia (Elbow Method):** Measures total within-cluster variance. Diminishing returns indicate the elbow point — adding more clusters beyond this provides marginal improvement at the cost of interpretability.

**Silhouette Score:** Measures how well-separated clusters are from each other. More rigorous than the elbow — a score close to 1 means tight, well-separated clusters. I use this as the primary selection criterion.

In [ ]:
K_range = range(2, 11)
inertias, silhouette_scores = [], []

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(X_pca)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_pca, labels, sample_size=3000, random_state=42)
    silhouette_scores.append(sil)
    print(f'K={k:2d}  |  Inertia: {km.inertia_:>10.0f}  |  Silhouette: {sil:.4f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(list(K_range), inertias, 'b-o', linewidth=2, markersize=8)
ax1.set_xlabel('K (Number of Clusters)')
ax1.set_ylabel('Inertia')
ax1.set_title('Elbow Curve')
ax1.grid(True, alpha=0.3)

best_k = list(K_range)[np.argmax(silhouette_scores)]
ax2.plot(list(K_range), silhouette_scores, 'r-o', linewidth=2, markersize=8)
ax2.axvline(x=best_k, color='green', linestyle='--',
            label=f'Optimal K={best_k}  (score={max(silhouette_scores):.4f})')
ax2.set_xlabel('K (Number of Clusters)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Score vs K')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('K Selection — Elbow and Silhouette Methods', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('cluster_selection.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Selected K = {best_k} based on maximum silhouette score.')

In [ ]:
OPTIMAL_K = best_k

# Final model — k-means++ initialisation, 20 restarts to avoid local minima
kmeans_final = KMeans(
    n_clusters=OPTIMAL_K, init='k-means++',
    n_init=20, max_iter=300, random_state=42
)
cluster_labels = kmeans_final.fit_predict(X_pca)

df_kpi['CLUSTER'] = cluster_labels
df_clean['CLUSTER'] = cluster_labels

final_sil = silhouette_score(X_pca, cluster_labels)
print(f'Final silhouette score: {final_sil:.4f}')
print('Cluster sizes:')
print(df_kpi['CLUSTER'].value_counts().sort_index())

In [ ]:
# 2D cluster visualisation
plot_df = pd.DataFrame({'PC1': X_2d[:, 0], 'PC2': X_2d[:, 1],
                        'Cluster': cluster_labels.astype(str)})

fig = px.scatter(
    plot_df, x='PC1', y='PC2', color='Cluster',
    title=f'Customer Segments — K={OPTIMAL_K} in PCA Space',
    opacity=0.6, width=900, height=600,
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig.update_traces(marker=dict(size=4))
fig.show()

---
## 8. Segment Profiling

Clusters are only useful once interpreted. I profile each segment across the key KPIs and original features to assign business-meaningful labels. The radar chart and heatmap make the inter-cluster differences immediately visible.

In [ ]:
profile_cols = [
    'BALANCE', 'PURCHASES', 'CASH_ADVANCE', 'CREDIT_LIMIT', 'PAYMENTS',
    'MINIMUM_PAYMENTS', 'PRC_FULL_PAYMENT', 'TENURE',
    'PURCHASE_TO_LIMIT', 'CASH_ADVANCE_RATIO',
    'PAYMENT_TO_MINPAYMENT', 'INSTALLMENT_RATIO', 'SPEND_PER_TRANSACTION'
]

cluster_profile = df_kpi.groupby('CLUSTER')[profile_cols].mean().round(2)
cluster_profile['SIZE']     = df_kpi.groupby('CLUSTER').size()
cluster_profile['SIZE_PCT'] = (cluster_profile['SIZE'] / len(df_kpi) * 100).round(1)

print('Cluster Mean Profiles:')
cluster_profile.T

In [ ]:
# Radar chart — normalised to 0-1 for visual comparison across features
radar_cols = ['BALANCE', 'PURCHASES', 'CASH_ADVANCE',
              'CREDIT_LIMIT', 'PAYMENTS', 'PRC_FULL_PAYMENT']

profile_norm = cluster_profile[radar_cols].copy()
for col in radar_cols:
    cmin, cmax = profile_norm[col].min(), profile_norm[col].max()
    if cmax > cmin:
        profile_norm[col] = (profile_norm[col] - cmin) / (cmax - cmin)

fig = go.Figure()
colors_radar = ['#636EFA','#EF553B','#00CC96','#AB63FA','#FFA15A','#19D3F3']

for k in range(OPTIMAL_K):
    vals = profile_norm.loc[k, radar_cols].tolist()
    fig.add_trace(go.Scatterpolar(
        r=vals + [vals[0]],
        theta=radar_cols + [radar_cols[0]],
        fill='toself', name=f'Cluster {k}',
        line_color=colors_radar[k % len(colors_radar)], opacity=0.6
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0,1])),
    title='Segment Profiles — Radar Chart',
    width=700, height=600
)
fig.show()

In [ ]:
# Heatmap — z-scored for visual contrast across clusters
plt.figure(figsize=(16, 5))
profile_z = cluster_profile[radar_cols].copy()
for col in radar_cols:
    profile_z[col] = (profile_z[col] - profile_z[col].mean()) / (profile_z[col].std() + 1e-9)

sns.heatmap(profile_z, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, annot_kws={'size': 10})
plt.title('Cluster Profiles — Z-Scored Feature Means', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('cluster_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Assign segment labels based on dominant characteristics
# Adjust these after reviewing the profile table above
cluster_names = {}
for k in range(OPTIMAL_K):
    row = cluster_profile.loc[k]
    med = cluster_profile.median()
    if row['PRC_FULL_PAYMENT'] > med['PRC_FULL_PAYMENT'] and row['PURCHASES'] > med['PURCHASES']:
        cluster_names[k] = f'Cluster {k}: Transactors'
    elif row['CASH_ADVANCE_RATIO'] > med['CASH_ADVANCE_RATIO']:
        cluster_names[k] = f'Cluster {k}: Cash-Advance Dependent'
    elif row['BALANCE'] > med['BALANCE'] and row['PRC_FULL_PAYMENT'] < med['PRC_FULL_PAYMENT']:
        cluster_names[k] = f'Cluster {k}: Revolvers'
    else:
        cluster_names[k] = f'Cluster {k}: Inactive / Low-Use'

print('Segment labels:')
for k, name in cluster_names.items():
    print(f'  {name}  —  {int(cluster_profile.loc[k,"SIZE"])} customers ({cluster_profile.loc[k,"SIZE_PCT"]}%)')

---
## 9. Per-Segment Anomaly Detection — Isolation Forest

**Design rationale for per-segment detection:**
Running a single global Isolation Forest treats the entire customer base as one distribution. A Revolver with high balance would score as anomalous relative to Inactive customers — which is a false positive. By running a separate Isolation Forest on each cluster, I detect customers who are anomalous *within their behavioural peer group*. This dramatically reduces false positives and makes the anomaly flags more actionable.

**Why Isolation Forest:**
Isolation Forest works by randomly partitioning the feature space with decision trees. Anomalous points — those far from the cluster's typical region — get isolated in fewer partitions than normal points packed together. The anomaly score is the inverse of the average path length to isolation. It's computationally efficient, handles high-dimensional data well, and doesn't require a distance metric.

In [ ]:
CONTAMINATION = 0.05  # expect ~5% anomaly rate per segment

df_kpi['ANOMALY']       = 0
df_kpi['ANOMALY_SCORE'] = 0.0

anomaly_results = {}

for cluster_id in range(OPTIMAL_K):
    cluster_mask    = df_kpi['CLUSTER'] == cluster_id
    cluster_indices = df_kpi[cluster_mask].index
    X_cluster       = X_kpi[cluster_indices]

    iso = IsolationForest(
        n_estimators=200,
        contamination=CONTAMINATION,
        max_samples='auto',
        random_state=42,
        n_jobs=-1
    )
    predictions = iso.fit_predict(X_cluster)   # 1 = normal, -1 = anomaly
    scores      = iso.score_samples(X_cluster) # lower = more anomalous

    df_kpi.loc[cluster_indices, 'ANOMALY']       = np.where(predictions == -1, 1, 0)
    df_kpi.loc[cluster_indices, 'ANOMALY_SCORE'] = scores

    n_anom  = (predictions == -1).sum()
    n_total = len(cluster_indices)
    anomaly_results[cluster_id] = {'n_total': n_total, 'n_anomalies': n_anom, 'model': iso}

    print(f'{cluster_names[cluster_id]}')
    print(f'  {n_total} customers  →  {n_anom} anomalies flagged ({n_anom/n_total*100:.1f}%)')

print(f'\nTotal anomalies: {df_kpi["ANOMALY"].sum()} / {len(df_kpi)} ({df_kpi["ANOMALY"].mean()*100:.1f}%)')

In [ ]:
# Visualise anomalies against cluster structure
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors_list = cm.get_cmap('tab10')(np.linspace(0, 1, OPTIMAL_K))

# Left: cluster view with anomalies marked
ax = axes[0]
for k in range(OPTIMAL_K):
    nm = (cluster_labels == k) & (df_kpi['ANOMALY'].values == 0)
    am = (cluster_labels == k) & (df_kpi['ANOMALY'].values == 1)
    ax.scatter(X_2d[nm, 0], X_2d[nm, 1], c=[colors_list[k]], s=8,  alpha=0.4, label=f'Cluster {k}')
    ax.scatter(X_2d[am, 0], X_2d[am, 1], c='red', marker='x', s=30, alpha=0.8, linewidths=1)
ax.set_title('Clusters with Anomalies Flagged (red ×)', fontsize=11, fontweight='bold')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.legend(markerscale=3, fontsize=8)

# Right: continuous anomaly score
ax2 = axes[1]
sc = ax2.scatter(X_2d[:, 0], X_2d[:, 1],
                 c=df_kpi['ANOMALY_SCORE'].values, cmap='RdYlGn', s=8, alpha=0.5)
plt.colorbar(sc, ax=ax2, label='Anomaly Score (lower = more anomalous)')
ax2.set_title('Continuous Anomaly Score', fontsize=11, fontweight='bold')
ax2.set_xlabel('PC1'); ax2.set_ylabel('PC2')

plt.suptitle('Anomaly Detection Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('anomaly_results.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# What distinguishes anomalous customers within each segment?
key_features = ['BALANCE', 'PURCHASES', 'CASH_ADVANCE', 'PAYMENTS',
                'PRC_FULL_PAYMENT', 'CREDIT_LIMIT', 'CASH_ADVANCE_RATIO', 'PAYMENT_TO_MINPAYMENT']

print('WITHIN-SEGMENT ANOMALY PROFILES')
print('='*70)

for k in range(OPTIMAL_K):
    seg   = df_kpi[df_kpi['CLUSTER'] == k]
    norm  = seg[seg['ANOMALY'] == 0]
    anom  = seg[seg['ANOMALY'] == 1]
    if len(anom) == 0: continue

    print(f'\n{cluster_names[k]}  |  {len(norm)} normal  /  {len(anom)} anomalous')
    print(f'  {"Feature":<30} {"Normal":>12} {"Anomalous":>12} {"Ratio":>8}')
    print('  ' + '-'*65)

    for feat in key_features:
        if feat not in df_kpi.columns: continue
        nm, am = norm[feat].mean(), anom[feat].mean()
        ratio  = am / nm if abs(nm) > 1e-6 else float('nan')
        flag   = '  ◄' if abs(ratio - 1) > 0.5 else ''
        print(f'  {feat:<30} {nm:>12.2f} {am:>12.2f} {ratio:>8.2f}x{flag}')

In [ ]:
# Box plots: normal vs anomalous per cluster across key features
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
plot_feats = ['BALANCE', 'PURCHASES', 'CASH_ADVANCE',
              'PAYMENTS', 'CREDIT_LIMIT', 'PRC_FULL_PAYMENT']

for idx, feat in enumerate(plot_feats):
    ax = axes[idx]
    data_plot, labels_plot = [], []
    for k in range(OPTIMAL_K):
        seg = df_kpi[df_kpi['CLUSTER'] == k]
        data_plot += [seg[seg['ANOMALY']==0][feat].values, seg[seg['ANOMALY']==1][feat].values]
        labels_plot += [f'C{k}\nNorm', f'C{k}\nAnom']

    bp = ax.boxplot(data_plot, labels=labels_plot, patch_artist=True, notch=False,
                    medianprops=dict(color='black', linewidth=2))
    for i, patch in enumerate(bp['boxes']):
        patch.set_facecolor('steelblue' if i % 2 == 0 else 'salmon')
        patch.set_alpha(0.7)
    ax.set_title(feat, fontsize=10, fontweight='bold')
    ax.tick_params(axis='x', labelsize=7)

plt.suptitle('Normal (Blue) vs Anomalous (Red) — Per Segment Per Feature',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('anomaly_boxplots.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 10. Results and Business Implications

In [ ]:
print('='*70)
print('SYSTEM OUTPUT SUMMARY')
print('='*70)
print(f'Customers processed   : {len(df_kpi)}')
print(f'Features (engineered) : {X_kpi.shape[1]} ({len(kpi_cols)} KPIs + {X_kpi.shape[1]-len(kpi_cols)} original)')
print(f'PCA components used   : {N_COMPONENTS} ({pca.explained_variance_ratio_.sum()*100:.1f}% variance)')
print(f'Segments identified   : {OPTIMAL_K}')
print(f'Silhouette score      : {final_sil:.4f}')
print(f'Anomalies flagged     : {df_kpi["ANOMALY"].sum()} ({df_kpi["ANOMALY"].mean()*100:.1f}%)')
print()
print('SEGMENT BREAKDOWN')
print('-'*70)
for k in range(OPTIMAL_K):
    row   = cluster_profile.loc[k]
    n_anom = df_kpi[(df_kpi['CLUSTER']==k) & (df_kpi['ANOMALY']==1)].shape[0]
    print(f'  {cluster_names[k]}')
    print(f'  {int(row["SIZE"])} customers ({row["SIZE_PCT"]}%)  |  '
          f'Avg balance: ${row["BALANCE"]:,.0f}  |  '
          f'Full-pay rate: {row["PRC_FULL_PAYMENT"]*100:.1f}%  |  '
          f'Anomalies: {n_anom}')
    print()

In [ ]:
# Summary dashboard
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
cluster_colors = cm.get_cmap('tab10')(np.linspace(0, 1, OPTIMAL_K))

sizes = [anomaly_results[k]['n_total'] for k in range(OPTIMAL_K)]
ax1.pie(sizes, labels=[f'Cluster {k}\n({sizes[k]})' for k in range(OPTIMAL_K)],
        autopct='%1.1f%%', colors=cluster_colors, startangle=90, pctdistance=0.85)
ax1.set_title('Customer Distribution by Segment', fontsize=12, fontweight='bold')

anom_pcts = [anomaly_results[k]['n_anomalies']/anomaly_results[k]['n_total']*100
             for k in range(OPTIMAL_K)]
bars = ax2.bar(range(OPTIMAL_K), anom_pcts, color=cluster_colors, edgecolor='white', linewidth=1.5)
ax2.set_xticks(range(OPTIMAL_K))
ax2.set_xticklabels([f'Cluster {k}' for k in range(OPTIMAL_K)])
ax2.set_ylabel('Anomaly Rate (%)')
ax2.set_title('Anomaly Rate per Segment', fontsize=12, fontweight='bold')
ax2.axhline(CONTAMINATION*100, color='red', linestyle='--', label='Target rate')
ax2.legend()
for bar, val in zip(bars, anom_pcts):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('summary_dashboard.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Export labelled dataset
output_df = df.copy()
if 'CUST_ID' in output_df.columns:
    output_df = output_df.set_index('CUST_ID')

output_df['CLUSTER']      = cluster_labels
output_df['SEGMENT']      = [cluster_names[k].split(':')[1].strip() for k in cluster_labels]
output_df['ANOMALY']      = df_kpi['ANOMALY'].values
output_df['ANOMALY_SCORE']= df_kpi['ANOMALY_SCORE'].values
for col in kpi_cols:
    output_df[col] = df_kpi[col].values

output_df.to_csv('customer_risk_profiles.csv')
print('Exported: customer_risk_profiles.csv')
print(f'{len(output_df)} customers  |  {len(output_df.columns)} columns')

---
## Segment Strategy Summary

| Segment | Risk Profile | Recommended Action |
|---|---|---|
| **Transactors** | Low credit risk, high revenue | Premium rewards, credit limit increases, cross-sell investment products |
| **Revolvers** | Medium risk, interest income source | Balance transfer offers, debt consolidation products, early-warning monitoring |
| **Cash-Advance Dependent** | High risk, liquidity stress signals | Personal loan alternatives, financial wellness outreach, tighter monitoring |
| **Inactive / Low-Use** | Dormancy risk, low revenue | Re-engagement campaigns, cashback on first spend, review for account closure |

**Anomaly action thresholds:**
- Score < −0.20 → Immediate review queue (potential fraud or distress)
- Score −0.20 to −0.10 → Weekly monitoring alert
- Score > −0.10 → Normal behaviour, no action